Hay que evaluar la performance de la inferencia. 

In [6]:
import pandas as pd
from pathlib import Path


# Cargar los comandos desde tu dataset limpio o de evaluación
#project_root = Path(__file__).resolve().parents[2]
project_root = Path.cwd().parents[1]
print(f"Project root: {project_root}")

data_path = project_root / "data/processed/shell_safety/test.jsonl"
df = pd.read_json(data_path, lines=True)

sample_rows = df[:10].to_dict(orient="records")
#df.head()

Project root: c:\Users\Usuario\Desktop\taed2-SafeAgent


In [30]:
import time
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from taed2_safeagent.config import MODEL_NAMES

model_name = MODEL_NAMES[0]
model_dir = project_root / Path(f"models/encoders/{model_name}")

# Cargar el tokenizador y el modelo desde la carpeta local
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

# Asegurar que corre en CPU y en modo evaluación (sin calcular gradientes)
model.to("cpu")
model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [31]:
from taed2_safeagent.data.inputs import build_input
from taed2_safeagent.modeling.common import load_config, load_examples

# Detectar el modo según el nombre del modelo
input_mode = "command-context" if "command-context" in model_name else "command"

# Transformar las filas del dataset al texto esperado por el modelo
data_sample = df[:10]
sample_texts = [build_input(row, input_mode) for row in sample_rows]


from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 1. Tiempo de carga
start_load = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).to("cpu")
model.eval()
load_time = time.perf_counter() - start_load

# 2. Inferencia (tokenizando los `sample_texts`)
inputs = tokenizer(
    sample_texts, padding=True, truncation=True, return_tensors="pt"
).to("cpu")

with torch.no_grad():
    outputs = model(**inputs)
    predictions = outputs.logits.argmax(dim=-1)